### MC Shapley + Sobol from a Single Data Collection

The Owen & Prieur (2017) formulation for correlated-input Shapley effects
uses the cost function:

$$v(u) = \text{Cov}[f(\mathbf{X}), f(\mathbf{X}_u)]$$

where $\mathbf{X}_u$ shares the same background variables as $\mathbf{X}$
but conditions on the subset $u$.  Remarkably, this same quantity equals
the **closed Sobol index**:

$$v(u) = \mathbb{V}[\mathbb{E}(f(\mathbf{X}) \mid \mathbf{X}_u)]$$

So from the **same Monte Carlo data** used for Shapley effects, we can
extract both first-order ($S_i$) and total-order ($T_i$) Sobol indices
at zero additional cost.

$$S_i = \frac{v(\{i\})}{v(\text{full})}, \qquad
T_i = 1 - \frac{v(\text{all}\\setminus\{i\})}{v(\text{full})}$$

This notebook demonstrates the `MCShapley.compute()` method which now
returns `sobol_first` and `sobol_total` columns alongside the Shapley
effects.

We use the **Ishigami function** with correlated inputs as the test case.

In [ ]:
# Import dependencies
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from shapleyx.utilities.mc_shapley import (
    GaussianCopulaUniform,
    MCShapley,
)

from importlib.metadata import version
print(f"ShapleyX v{version('shapleyx')}")

In [ ]:
def ishigami(x):
    """Ishigami function (1D input)."""
    return np.sin(x[0]) + 7.0 * np.sin(x[1])**2 + 0.1 * x[2]**4 * np.sin(x[0])


def ishigami_batch(X):
    """Vectorised Ishigami for batch evaluation."""
    x1, x2, x3 = X[:, 0], X[:, 1], X[:, 2]
    return np.sin(x1) + 7.0 * np.sin(x2)**2 + 0.1 * x3**4 * np.sin(x1)


d = 3
print(f"Ishigami function ready (d = {d})")

---
### Correlated Input Distribution

Uniform marginals on $[-\pi, \pi]$ with $X_1$–$X_3$ correlated
($\rho = 0.8$) via a Gaussian copula.

In [ ]:
corr = np.array([
    [1.0, 0.0, 0.8],
    [0.0, 1.0, 0.0],
    [0.8, 0.0, 1.0],
])

joint = GaussianCopulaUniform(
    lows=[-np.pi] * d,
    highs=[np.pi] * d,
    corr=corr,
)
print(f"GaussianCopulaUniform: d = {joint.d}")

---
### MC Shapley + Sobol in One Call

`MCShapley.compute()` with `method='exhaustive'` now returns a DataFrame
with `sobol_first` and `sobol_total` columns — no extra model evaluations.

In [ ]:
mc = MCShapley(f=ishigami, joint=joint, predict_batch=ishigami_batch)

results = mc.compute(
    N=5000,
    method='exhaustive',
    B=200,
    alpha=0.05,
    random_state=42,
    progress=True,
)
results

---
### Visual Comparison: Shapley vs Sobol

For correlated inputs, the relationship between indices becomes
informative:
- **$S_i$ (first-order)**: main effect of $X_i$ alone — ignores interactions
- **Shapley**: fairly distributes interaction variance among participants
- **$T_i$ (total-order)**: effect of $X_i$ including all interactions

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(d)
bar_width = 0.25

ax.bar(x - bar_width, results['sobol_first'], bar_width,
       color='steelblue', alpha=0.85, label='First-order Sobol $S_i$')
ax.bar(x, results['effect'], bar_width,
       yerr=[results['effect'] - results['lower'],
             results['upper'] - results['effect']],
       capsize=5, color='darkorange', alpha=0.85, label='Shapley effect')
ax.bar(x + bar_width, results['sobol_total'], bar_width,
       color='seagreen', alpha=0.85, label='Total-order Sobol $T_i$')

ax.set_xlabel('Input Variable')
ax.set_ylabel('Sensitivity Index')
ax.set_title('Shapley vs Sobol — Ishigami Function\n(Correlated: $\\rho_{13} = 0.8$)')
ax.set_xticks(x)
ax.set_xticklabels(results['variable'])
ax.legend(fontsize=9)
ax.set_ylim(0, None)
plt.tight_layout()
plt.show()

---
### Interpreting the Gaps

The differences between $S_i$, Shapley, and $T_i$ reveal how interactions
are distributed under correlation:

| Pattern | Interpretation |
|---|---|
| $S_i \approx$ Shapley $\approx T_i$ | $X_i$ has no interactions with other inputs |
| $S_i <$ Shapley $< T_i$ | $X_i$ participates in interactions — Shapley splits the interaction variance |
| $T_i - S_i$ large | Strong interactions involving $X_i$ |

Here, $X_3$ shows the largest $T_i - S_i$ gap, reflecting its strong
interaction with $X_1$ in the Ishigami function ($0.1 x_3^4 \sin(x_1)$).
The correlation ($\\rho_{13}=0.8$) amplifies this shared effect.

---
### Key Takeaways

1. **`MCShapley.compute()` now returns Sobol indices** in the same
   DataFrame — use `method='exhaustive'` for `sobol_first` and
   `sobol_total` columns.
2. **Zero additional cost** — Sobol indices are extracted from the
   same $v(u)$ values computed for Shapley effects.
3. **Shapley sits between $S_i$ and $T_i$** — it fairly distributes
   interaction variance, making it interpretable as a single-number
   summary of each input's total importance.
4. **The gaps reveal interactions** — large $(T_i - S_i)$ indicates
   strong interactions; differences under correlation also reflect
   shared information between correlated inputs.

In [ ]:
%load_ext watermark
%watermark -n -u -v -iv -w